# This notebook uses the baseline model of the competition as a starting point. The goal is to improve the metric score. The repository of the competition can be found [here](https://github.com/royerlab/kaggle-cell-tracking-competition/tree/main/tests/assets/sandbox_examples)

# Defining input data

In [1]:
from pathlib import Path
import os
DATA_PATH = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAINING_PATH =  DATA_PATH/'train'
TEST_PATH = DATA_PATH/'test'


#------------ Listing folders in train -----------
print("Input contents:", os.listdir(TRAINING_PATH)[:5],"\n")
print(" First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)")

Input contents: ['6bba_2540cd90.geff', '44b6_0b24845f.geff', '44b6_996155de.geff', '44b6_0c582fdc.geff', '6bba_cf35214c.zarr'] 

 First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)


# Grid search
we are going to split the grid search into two sub-sets according how expensive is to vary its value.

**Set 1**: consist of the cheapest variables, it means, changing its value doesn't mean a huge computational cost. This is because they use the same data, same model shape, and only the optimizer/loss behavior changes. These variables are:
- lr
- det_loss_weight
- det_neg_weight
- det_threshold

**Set 2**: The variables are
- pool_kernel_um
- window_size

Additionally, as we want to have an intuition and idea about how much the model improves by variying these values, we won't expand the grid search for houndreds of elements. Instead, we are going to consider a small array of values as starting point to see how the model resopnds. Initiallz we are going to consider:
- n_epochs = 5
- max_iter = 300

## Modification to baseline model

The ```train()``` function in the baseline model **doesn't return metrics**, it only returns the model, for this reason we need to change its output to

``` 
return model, {"best_score": best_score, "final_edge_loss": edge_loss,
                "final_det_loss": det_loss, "final_test_acc": test_acc,
                "final_test_recall": test_recall}
```

Additionally, **det_threshold** is currently a constant (set to 0.3), we want to explore its effect so we neew to modify the code and turn it into a input variable.

In [2]:
import shutil, sys
from pathlib import Path
# --------- Making a copy of the folder to my own working space --------------
# ---------             such that I can edit it     --------------------------

ARTIFACTS_SRC  = Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts")
ARTIFACTS_WORK = Path("/kaggle/working/cellmot-baseline-artifacts")

if not ARTIFACTS_WORK.exists():
    shutil.copytree(ARTIFACTS_SRC, ARTIFACTS_WORK)

# put the writable copy ahead of anything else on the path
sys.path.insert(0, str(ARTIFACTS_WORK))
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/scripts"))   # adjust to wherever train_unet_transformer.py actually sits
sys.path.insert(0, str(ARTIFACTS_WORK / "repo/src"))

In [3]:
needed_dirs = {
    matches[0].parent for name in
    ("train_unet_transformer.py", "tracking_cellmot", "augmentations.py", "dataspec.py")
    if (matches := list(ARTIFACTS_WORK.rglob(name)))
}
needed_dirs

{PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/scripts'),
 PosixPath('/kaggle/working/cellmot-baseline-artifacts/repo/src')}

### Intalling dependencies

In [4]:
import glob
import os
import subprocess
import sys
import shutil

# 1. Find all wheels, but filter OUT numpy wheels to avoid breaking C-extensions
wheels = [
    f for f in glob.glob(f"{ARTIFACTS_SRC}/wheels/*.whl")
    if "numpy" not in os.path.basename(f).lower()
]

# 2. Install only the required non-NumPy wheels without upgrading dependencies
subprocess.run(
    [
        "pip", "install", 
        "--no-index", 
        "--find-links", f"{ARTIFACTS_SRC}/wheels",
        "--no-deps",
        *wheels
    ],
    check=True)

Looking in links: /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-bas

CompletedProcess(args=['pip', 'install', '--no-index', '--find-links', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels', '--no-deps', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl', '/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl', '/kaggle/input/datasets/thibautgolds

### Creating `splits_file`file for the training and test datasets

In [5]:
import json
import random
from pathlib import Path

if not TRAINING_PATH.is_dir():
    raise FileNotFoundError(f"TRAINING_PATH does not exist or is not a directory: {TRAINING_PATH}")

# stems with matching .geff annotation, same convention as the rest of the pipeline
train_pool_stems = sorted(
    p.stem for p in TRAINING_PATH.iterdir()
    if p.is_dir() and p.suffix == ".zarr"
    and (TRAINING_PATH / f"{p.stem}.geff").exists()
)
print(f"{len(train_pool_stems)} videos available in training pool")


MAX_SAMPLES = 10 # define how many dataset will be considered MAX_SAMPLES <= len(train_pool_stems)
VAL_FRACTION = 0.15  # 15% of the total considered data 
n_val = max(1, round(MAX_SAMPLES * VAL_FRACTION))

rng = random.Random(0)          # fixed seed — same split reused across every hp-search trial
shuffled = train_pool_stems.copy()
rng.shuffle(shuffled)
subset = shuffled[:MAX_SAMPLES]

val_stems = sorted(subset[:n_val])
train_stems = sorted(subset[n_val:])

print(f"{len(train_stems)} train / {len(val_stems)} val")
assert set(train_stems).isdisjoint(val_stems)  # sanity check — no leakage between the two

splits = [{"split": 0, "train": train_stems, "test": val_stems}]
# if split = "all" the "train()" function performs 5 folds during training
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

199 videos available in training pool
8 train / 2 val


In [6]:
#---------- Reading back the json file just created ------------
with open(f"{ARTIFACTS_WORK}/kaggle_train_val_splits.json", "r") as f:
    json_file = json.load(f)

json_file

[{'split': 0,
  'train': ['44b6_1574802b',
   '44b6_d5e7d891',
   '6bba_2312ac41',
   '6bba_5c824876',
   '6bba_7af54fde',
   '6bba_7b5d3b2c',
   '6bba_afb141ff',
   '6bba_d1acb6ff'],
  'test': ['44b6_d754aa59', '6bba_268e1230']}]

### Applying modifications to the baseline code

Working now on the folder located in my `work` folder

In [7]:
REPO_DIR = ARTIFACTS_WORK / "repo"
target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # path to the training script
src = target.read_text()

# 1. thread det_threshold through train_epoch
src = src.replace(
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n) -> tuple[float, float]:",
    "def train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n    det_threshold: float = 0.3,\n) -> tuple[float, float]:"
)
src = src.replace(
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction",
    "                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                det_threshold=det_threshold,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction"
)

# 2. train() returns metrics instead of just the model
src = src.replace(
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    if save_path.exists():",
    "    print(f\"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}\")\n    metrics = {\"best_score\": best_score, \"final_edge_loss\": edge_loss,\n               \"final_det_loss\": det_loss, \"final_test_acc\": test_acc,\n               \"final_test_recall\": test_recall}\n    if save_path.exists():"
)
src = src.replace(
    "        model.load_state_dict(state)\n    return model",
    "        model.load_state_dict(state)\n    return model, metrics"
)

target.write_text(src)

49486

In [8]:
REPO_DIR = ARTIFACTS_WORK / "repo"

target = REPO_DIR / "scripts" / "train_unet_transformer.py"  # same path as before
src = target.read_text()

# 1. add full_checkpoint to train()'s signature
src = src.replace(
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n) -> UNetNodeTransformer:",
    "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n    full_checkpoint: Path | None = None,\n    det_threshold: float = 0.3,\n) -> UNetNodeTransformer:"
)

# 2. load it right after model construction, BEFORE any DataParallel wrapping
#    (checkpoint keys are unwrapped "unet.*", wrapping would change them to "unet.module.*")
src = src.replace(
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n",
    "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n\n"
    "    if full_checkpoint is not None:\n"
    "        ckpt_state = torch.load(full_checkpoint, map_location=device, weights_only=True)\n"
    "        missing, unexpected = model.load_state_dict(ckpt_state, strict=False)\n"
    "        print(f\"  Full checkpoint loaded from {full_checkpoint}: \"\n"
    "              f\"{len(missing)} missing, {len(unexpected)} unexpected\", flush=True)\n"
    "        if missing or unexpected:\n"
    "            print(f\"    missing (sample): {missing[:5]}\", flush=True)\n"
    "            print(f\"    unexpected (sample): {unexpected[:5]}\", flush=True)\n"
)

# 2. pass it into the train_epoch(...) call inside the epoch loop
src = src.replace(
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um,\n"
    "        )",
    "        edge_loss, det_loss = train_epoch(\n"
    "            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n"
    "            max_iters=max_iters, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n"
    "        )"
)

# 3. pass it into the evaluate(...) call inside the epoch loop
src = src.replace(
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)",
    "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)"
)


target.write_text(src)

# sanity checks — confirm both edits actually matched before trusting them
assert "full_checkpoint: Path | None = None," in src
assert "Full checkpoint loaded from" in src
assert "det_threshold: float = 0.3,\n) -> UNetNodeTransformer:" in src
assert "pool_kernel_um=pool_kernel_um, det_threshold=det_threshold,\n        )" in src
assert "evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um, det_threshold=det_threshold)" in src

In [9]:
# 1. add det_threshold to evaluate()'s signature
src = src.replace(
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    ") -> tuple[float, float, float]:",
    "def evaluate(\n"
    "    model: UNetNodeTransformer,\n"
    "    loader: DataLoader,\n"
    "    device: torch.device,\n"
    "    pool_kernel_um: float = 5.0,\n"
    "    det_threshold: float = 0.3,\n"
    ") -> tuple[float, float, float]:"
)

# 2. pass it into detect_and_match(...) inside evaluate()'s loop
src = src.replace(
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:",
    "            det_c, det_p, det_m, matches = detect_and_match(\n"
    "                det_logits[i], coords[:, i], masks[:, i],\n"
    "                image_shape,\n"
    "                voxel_size=voxel_size,\n"
    "                pool_kernel_um=pool_kernel_um,\n"
    "                det_threshold=det_threshold,\n"
    "                frame_index=i, window_size=W,\n"
    "            )\n"
    "            unet_feat = model._index_features(\n"
    "                unet_out[:, i], det_c, det_m,\n"
    "            )\n"
    "            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n"
    "\n"
    "            # Node recall:"
)

target.write_text(src)
assert "det_threshold: float = 0.3,\n) -> tuple[float, float, float]:" in src

In [10]:
lines = target.read_text().splitlines(keepends=True)

# line 1201 in your grep output is 1-indexed; list index is 1200
assert "Best score (acc" in lines[1200], f"unexpected content at index 1200: {lines[1200]!r}"

metrics_block = (
    '    metrics = {"best_score": best_score, "final_edge_loss": edge_loss,\n'
    '               "final_det_loss": det_loss, "final_test_acc": test_acc,\n'
    '               "final_test_recall": test_recall}\n'
)

lines.insert(1201, metrics_block)  # insert right after the print line
target.write_text("".join(lines))
print("inserted")

inserted


# Configuration

In [11]:


METHOD = "unet_transformer"

# --------------- Reading weights from the baseline model ---------------------------------------
# in the test set, the baseline model scored 0.8, let's see if we can improve this
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"


# Set 1 sweep

In [12]:
import itertools
import json
import time
from pathlib import Path
import gc
import torch
import random

from train_unet_transformer import train

RESULTS_PATH = Path("hp_search_results.jsonl")
random.seed(0)
n_random_trials = 24

def log_result(record: dict) -> None:
    with open(RESULTS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

def run_trial(trial_id: str, **kwargs) -> dict:
    t0 = time.monotonic()
    try:
        model, metrics = train(**kwargs)
    finally:
        # ensure cleanup happens even if this trial itself OOMs
        if "model" in dir():
            del model
        gc.collect()
        torch.cuda.empty_cache()
    elapsed = time.monotonic() - t0
    record = {
        "trial_id": trial_id,
        "elapsed_s": round(elapsed, 1),
        **{k: (str(v) if isinstance(v, Path) else v) for k, v in kwargs.items()
           if k not in ("data_dir", "splits_file", "unet_layers", "full_checkpoint")},
        **metrics,
    }
    log_result(record)
    print(f"[{trial_id}] score={metrics['best_score']:.4f}  ({elapsed:.0f}s)")
    return record


FULL_CHECKPOINT = ARTIFACTS_SRC / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
assert FULL_CHECKPOINT.exists(), f"not found: {FULL_CHECKPOINT}"

FIXED = dict(
    data_dir=Path(DATA_PATH)/"train",
    splits_file=ARTIFACTS_WORK/"kaggle_train_val_splits.json",
    fold=0,
    n_epochs=4,
    max_iters=200,
    seed=0,
    data_parallel=False,
    batch_size=2,
    method="hp_search_tier1",
    full_checkpoint=FULL_CHECKPOINT,   # <-- warm-start every trial from the 0.80 baseline
)

tier1_space = {
    "lr":              [3e-5, 1e-4, 3e-4],
    "det_loss_weight": [1e0, 3e0, 1e1],
    "det_neg_weight":  [3e-3, 1e-2, 3e-2],
    "det_threshold":   [0.40, 0.80, 0.99],
}

keys = list(tier1_space.keys())
combos = list(itertools.product(*tier1_space.values()))
random.shuffle(combos)


#for i, combo in enumerate(itertools.product(*tier1_space.values())):
for i, combo in enumerate(combos[:n_random_trials]):
    kwargs = dict(zip(keys, combo))
    run_trial(f"tier1_{i:03d}", **FIXED, **kwargs)
    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
    print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
    #break

Fold 0: 8 train, 2 test
Loading train (8 datasets)...


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)
train: 100%|██████████| 8/8 [00:05<00:00,  1.41it/s]

  train done: 787 windows total
Loading test (2 datasets)...



test: 100%|██████████| 2/2 [00:01<00:00,  1.79it/s]

  test done: 164 windows total
max_nodes=17


...

Training:  75%|███████▌  | 3/4 [18:33<04:39, 279.27s/it, acc=0.9973, det=0.0090, edge=0.0009]

  Epoch   3/4 | edge=0.0009 | det=0.0090 | test_loss=0.0024 | acc=0.9973 | recall=0.8749 | best=0.8806   | train=255.8s test=22.4s


Training: 100%|██████████| 4/4 [18:33<00:00, 278.49s/it, acc=0.9973, det=0.0090, edge=0.0009]


Best score (acc*recall): 0.8806, saved to /kaggle/working/cellmot-baseline-artifacts/repo/weights/hp_search_tier1/split_0/edge_predictor_best.pth


[tier1_023] score=0.8806  (1121s)
0.019136512 GB allocated
0.186646528 GB reserved


### Analysing outputs from set 1

In [13]:
import pandas as pd

df = pd.read_json("hp_search_results.jsonl", lines=True)
df = df.sort_values("best_score", ascending=False)
print(df[["trial_id", "lr", "det_loss_weight", "det_neg_weight",
          "det_threshold", "best_score", "final_test_recall"]].head(10))

# sanity-check marginal effect of each param independently
for param in ["lr", "det_loss_weight", "det_neg_weight", "det_threshold"]:
    print(df.groupby(param)["best_score"].agg(["mean", "std", "count"]))

     trial_id       lr  det_loss_weight  det_neg_weight  det_threshold  \
13  tier1_013  0.00003                1           0.030           0.99   
21  tier1_021  0.00003                3           0.010           0.99   
4   tier1_004  0.00003                3           0.003           0.40   
14  tier1_014  0.00003               10           0.010           0.99   
18  tier1_018  0.00003                3           0.003           0.80   
11  tier1_011  0.00003                3           0.003           0.99   
19  tier1_019  0.00010                1           0.030           0.80   
8   tier1_008  0.00010                3           0.003           0.80   
0   tier1_000  0.00003               10           0.030           0.40   
12  tier1_012  0.00010                3           0.030           0.40   

    best_score  final_test_recall  
13    0.928939           0.886219  
21    0.927859           0.913781  
4     0.922657           0.923675  
14    0.916977           0.901060  
18   